In [2]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'
MINCUBESAMPLES = 100
TARGETNAME = 'sr_all_eq'

SRFUNCTIONS = {
    'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
    'sqrt':np.sqrt,'exp':np.exp,'log':np.log,'abs':np.abs,
    'sin':np.sin,'cos':np.cos,'max':np.maximum,'min':np.minimum,
    '_safepow':lambda a,b:np.abs(a)**b}

import re
def _prepare_form(form):
    return re.sub(r'(\w+)\^(\w+)',r'_safepow(\1,\2)',form)

def eval_form(form,columns,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    ns.update(columns)
    ns.update(constants)
    out = eval(_prepare_form(form),ns)
    if np.ndim(out)==0:
        n = len(next(v for v in columns.values() if hasattr(v,'__len__')))
        out = np.full(n,float(out))
    return np.asarray(out,dtype=float)

In [4]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS['tp_mean']
STD  = STATS['tp_std']
ZMIN = (0.0 - MEAN) / STD

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds['dsig'].values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lfraw = flat('lf')
    shfraw = flat('shf')
    lhfraw = flat('lhf')
    blraw = flat('bl') if 'bl' in ds else np.zeros(ntime*ds.sizes['lat']*ds.sizes['lon'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsraw = ds['tp'].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)
meankernel = np.mean(kernels,axis=0)
weighted = fields * meankernel[None,:,:] * dsig[None,None,:]
if surfmask is not None:
    weighted = weighted * surfmask[:,None,:]
integrals = weighted.sum(axis=2)
rhraw,thetaeraw,thetaestarraw = integrals[:,0],integrals[:,1],integrals[:,2]

valid = np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw) & np.isfinite(obsraw)
rh,thetae,thetaestar = rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf,shf,lhf = lfraw[valid],shfraw[valid],lhfraw[valid]
bl = blraw[valid]
obs = obsraw[valid]
landmask  = lf > 0.5
oceanmask = lf < 0.5
print(f'Loaded {valid.sum():,} valid samples ({landmask.sum():,} land, {oceanmask.sum():,} ocean)')

Loaded 1,437,408 valid samples (428,352 land, 1,009,056 ocean)


In [5]:
regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in regdf.iterrows()}
SRMODELS = CONFIGS['experiments']['sr']['optimizedeqs']
ORDER  = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}
COLORS = {name:SRMODELS[name]['color'] for name in ORDER}

def get_columns(**overrides):
    cols = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
            'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    cols.update(overrides)
    for eqname,entry in REGISTRY.items():
        if eqname in overrides:
            continue
        cols[eqname] = eval_form(entry['form'],cols,entry['constants'])
    return cols

def predict_eq(name,columns):
    entry = REGISTRY[name]
    raw = eval_form(entry['form'],columns,entry['constants'])
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

cols = get_columns()
pred = predict_eq(TARGETNAME,cols)

r2all   = 1 - np.mean((pred - obs)**2) / np.var(obs)
r2land  = 1 - np.mean((pred[landmask] - obs[landmask])**2) / np.var(obs[landmask])
r2ocean = 1 - np.mean((pred[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])
print(f'{LABELS[TARGETNAME]} (unconstrained):')
print(f'  R\u00b2 all={r2all:.4f}  land={r2land:.4f}  ocean={r2ocean:.4f}')

SR-ALL (unconstrained):
  R² all=0.4149  land=0.2926  ocean=0.5089


In [ ]:
CONSTRAINTS = {
    'PC2':{
        'label':r'$\partial P/\partial \widehat{\mathrm{RH}} \geq 0$',
        'target_var':'rh',
        'expected_sign':1},
    'PC3':{
        'label':r'$\partial P/\partial \widehat{\theta_e} \geq 0$',
        'target_var':'thetae',
        'expected_sign':1},
    'PC4':{
        'label':r'$\partial P/\partial \widehat{\theta_e^*} \leq 0$',
        'target_var':'thetaestar',
        'expected_sign':-1}}

def calc_dz_srall(constants):
    '''Analytical dz/dx_i for SR-ALL (unconstrained).'''
    atm = REGISTRY['sr_atm_eq']['constants']
    a_atm,b_atm,c_atm = atm['a'],atm['b'],atm['c']
    arg = thetae - b_atm*thetaestar - c_atm
    rh_branch = rh >= arg
    buoy_branch = ~rh_branch
    dz_drh = np.where(rh_branch,3*a_atm*rh**2,0.0)
    dz_dthetae = np.where(buoy_branch,3*a_atm*arg**2,0.0) + (constants['b']-lf)**3
    dz_dthetaestar = np.where(buoy_branch,-b_atm*3*a_atm*arg**2,0.0)
    return {'rh':dz_drh,'thetae':dz_dthetae,'thetaestar':dz_dthetaestar}

derivs = calc_dz_srall(entry['constants'])
results = {}
for pcname,pc in CONSTRAINTS.items():
    dz = derivs[pc['target_var']]
    if pc['expected_sign'] >= 0:
        results[pcname] = {
            'Land':np.mean(dz[landmask] >= 0)*100,
            'Ocean':np.mean(dz[oceanmask] >= 0)*100,
            'All':np.mean(dz >= 0)*100}
    else:
        results[pcname] = {
            'Land':np.mean(dz[landmask] <= 0)*100,
            'Ocean':np.mean(dz[oceanmask] <= 0)*100,
            'All':np.mean(dz <= 0)*100}
    print(f'{pcname} ({pc["label"]}): Land={results[pcname]["Land"]:.1f}% '
          f'Ocean={results[pcname]["Ocean"]:.1f}% All={results[pcname]["All"]:.1f}%')

In [ ]:
entry = REGISTRY[TARGETNAME]
atmentry = REGISTRY['sr_atm_eq']

print(f'SR-ALL form:     {entry["form"]}')
print(f'SR-ALL constants: {entry["constants"]}')
print(f'SR-ATM form:     {atmentry["form"]}')
print(f'SR-ATM constants: {atmentry["constants"]}')
print()

# SR-ALL = a_atm * cube(max(rh, thetae - b_atm*thetaestar - c_atm))
#        + (thetae + a*shf) * cube(b - lf)
#        + c
#
# The max() creates two branches:
#   RH branch:       rh >= thetae - b_atm*thetaestar - c_atm
#   Buoyancy branch: rh <  thetae - b_atm*thetaestar - c_atm
#
# --- PC2: dz/d(rh) >= 0 ---
#   RH branch:       3*a_atm*rh^2                       >= 0 if a_atm > 0  CHECK
#   Buoyancy branch: 0                                   >= 0              CHECK
#
# --- PC3: dz/d(thetae) >= 0 ---
#   RH branch:       cube(b - lf)                        SIGN DEPENDS ON b VS lf
#   Buoyancy branch: 3*a_atm*(...)^2 + cube(b-lf)       FIRST TERM >= 0, SECOND UNCERTAIN
#
# --- PC4: dz/d(thetaestar) <= 0 ---
#   RH branch:       0                                   <= 0              CHECK
#   Buoyancy branch: -b_atm * 3*a_atm*(...)^2            <= 0 if a_atm,b_atm > 0  CHECK
#
# PC3 is the only analytically uncertain constraint.
# The violation comes from cube(b - lf), which is NEGATIVE when lf > b.

print('Verify a_atm > 0 and b_atm > 0 (required for PC2 and PC4):')
print(f'  a_atm = {atmentry["constants"]["a"]:.4f}  ({"OK" if atmentry["constants"]["a"] > 0 else "PROBLEM"})')
print(f'  b_atm = {atmentry["constants"]["b"]:.4f}  ({"OK" if atmentry["constants"]["b"] > 0 else "PROBLEM"})')
print()
print(f'SR-ALL correction constant b = {entry["constants"]["b"]:.4f}')
print(f'  Over ocean (lf ~ 0): cube({entry["constants"]["b"]:.2f} - 0) = {(entry["constants"]["b"])**3:.4f}')
print(f'  Over land  (lf ~ 1): cube({entry["constants"]["b"]:.2f} - 1) = {(entry["constants"]["b"] - 1)**3:.4f}')
print()
print('PC3 violation mechanism:')
print('  When lf > b, cube(b-lf) < 0, making dz/d(thetae) < 0 in the RH branch.')
print('  In the buoyancy branch, the positive 3*a_atm*(...)^2 term can compensate,')
print('  but not always — hence partial violations.')

In [ ]:
# Structurally modify SR-ALL so that PC3 (dz/d(thetae) >= 0) is guaranteed.
#
# The fix: subtract cube(b-1)*thetae from SR-ALL.
#
# Since LF in [0,1], min(cube(b-lf)) = cube(b-1). Subtracting this shifts the
# thetae coefficient from cube(b-lf) to cube(b-lf) - cube(b-1), which is >= 0
# for all LF in [0,1]:
#   At LF=0: cube(b) - cube(b-1) > 0
#   At LF=1: cube(b-1) - cube(b-1) = 0
#
# This guarantees dz/d(thetae) >= 0 in both branches:
#   RH branch:       cube(b-lf) - cube(b-1) >= 0                         CHECK
#   Buoyancy branch: 3*a_atm*(...)^2 + cube(b-lf) - cube(b-1) >= 0      CHECK
#                    (sum of non-negative terms)
#
# Advantages over max clipping:
#   - Smooth (no discontinuous derivative at lf = b)
#   - Preserves the cubic structure
#   - Over land where LF -> 1, the thetae correction smoothly vanishes
#     while the SHF*cube(b-lf) term can still contribute

PCFORM = 'sr_atm_eq+(thetae+a*shf)*cube(b-lf)-cube(b-1)*thetae+c'
PCNAME = 'sr_all_pc_eq'
PCLABEL = 'SR-ALL-PC'

print(f'Original:    {entry["form"]}')
print(f'Constrained: {PCFORM}')
print(f'Modification: thetae*cube(b-lf) -> thetae*(cube(b-lf) - cube(b-1))')
print()
print(f'Initial constants from SR-ALL: {entry["constants"]}')

In [9]:
from scipy.optimize import minimize

y = (np.log1p(obs) - MEAN) / STD
cols = get_columns()

constantnames = ['a','b','c']
init_all = entry['constants']
initparams = np.array([init_all[c] for c in constantnames])

def objective(params):
    constants = dict(zip(constantnames,params))
    raw = eval_form(PCFORM,cols,constants)
    pred = ZMIN + np.maximum(raw,0.0)
    return float(np.mean((pred - y)**2))

nrestarts = 50
rng = np.random.default_rng(42)
allresults = []
allinits = [initparams] + [rng.uniform(-5,5,len(constantnames)) for _ in range(nrestarts-1)]
for i,x0 in enumerate(allinits):
    res = minimize(objective,x0,method='L-BFGS-B',options={'maxiter':10000,'ftol':1e-14,'gtol':1e-10})
    allresults.append(res)
bestres = min(allresults,key=lambda r:r.fun)
pcconstants = dict(zip(constantnames,bestres.x))
pcconstants_rounded = {k:round(float(v),2) for k,v in pcconstants.items()}

print(f'Optimized on {SPLIT} split ({valid.sum():,} samples, {nrestarts} restarts)')
print(f'  Constants (raw):     {", ".join(f"{k}={v:.6f}" for k,v in pcconstants.items())}')
print(f'  Constants (rounded): {", ".join(f"{k}={v:.2f}" for k,v in pcconstants_rounded.items())}')
print(f'  MSE (z-space): {bestres.fun:.6f}  (SR-ALL unconstrained: {objective(initparams):.6f})')
print()
print(f'NOTE: These constants are optimized on the {SPLIT} split only.')
print('For final constants, run on train+valid via the pipeline:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq')

Optimized on test split (1,437,408 samples, 50 restarts)
  Constants (raw):     a=5.193709, b=0.730000, c=-0.098507
  Constants (rounded): a=5.19, b=0.73, c=-0.10
  MSE (z-space): 0.415095  (SR-ALL unconstrained: 0.415338)

NOTE: These constants are optimized on the test split only.
For final constants, run on train+valid via the pipeline:
  python -m scripts.models.sr.optimize --equations sr_all_pc_eq


In [ ]:
def predict_pc(columns):
    raw = eval_form(PCFORM,columns,pcconstants_rounded)
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

def calc_dz_pc(constants):
    '''Analytical dz/dx_i for SR-ALL-PC (constrained).'''
    atm = REGISTRY['sr_atm_eq']['constants']
    a_atm,b_atm,c_atm = atm['a'],atm['b'],atm['c']
    arg = thetae - b_atm*thetaestar - c_atm
    rh_branch = rh >= arg
    buoy_branch = ~rh_branch
    dz_drh = np.where(rh_branch,3*a_atm*rh**2,0.0)
    dz_dthetae = np.where(buoy_branch,3*a_atm*arg**2,0.0) + (constants['b']-lf)**3 - (constants['b']-1)**3
    dz_dthetaestar = np.where(buoy_branch,-b_atm*3*a_atm*arg**2,0.0)
    return {'rh':dz_drh,'thetae':dz_dthetae,'thetaestar':dz_dthetaestar}

derivs_pc = calc_dz_pc(pcconstants_rounded)
pcresults = {}
for pcname,pc in CONSTRAINTS.items():
    dz = derivs_pc[pc['target_var']]
    if pc['expected_sign'] >= 0:
        pcresults[pcname] = {
            'Land':np.mean(dz[landmask] >= 0)*100,
            'Ocean':np.mean(dz[oceanmask] >= 0)*100,
            'All':np.mean(dz >= 0)*100}
    else:
        pcresults[pcname] = {
            'Land':np.mean(dz[landmask] <= 0)*100,
            'Ocean':np.mean(dz[oceanmask] <= 0)*100,
            'All':np.mean(dz <= 0)*100}

print(f'Constraint satisfaction (analytical derivatives, all {SPLIT} samples):')
print(f'{"":>6} {"":>38} {"SR-ALL":>10} {"SR-ALL-PC":>10} {"Delta":>8}')
for pcname in CONSTRAINTS:
    label = CONSTRAINTS[pcname]['label']
    for region in ['Land','Ocean','All']:
        old = results[pcname][region]
        new = pcresults[pcname][region]
        delta = new - old
        tag = f'{pcname} {region}'
        print(f'{tag:>18} {label:>22}  {old:>8.1f}%  {new:>8.1f}%  {delta:>+7.1f}%')

In [11]:
predpc = predict_pc(cols)

r2pc_all   = 1 - np.mean((predpc - obs)**2) / np.var(obs)
r2pc_land  = 1 - np.mean((predpc[landmask] - obs[landmask])**2) / np.var(obs[landmask])
r2pc_ocean = 1 - np.mean((predpc[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])

print(f'Accuracy comparison ({SPLIT} split):')
print(f'{"Model":<14} {"R² all":>8} {"R² land":>9} {"R² ocean":>10}')
print(f'{"SR-ALL":<14} {r2all:>8.4f} {r2land:>9.4f} {r2ocean:>10.4f}')
print(f'{"SR-ALL-PC":<14} {r2pc_all:>8.4f} {r2pc_land:>9.4f} {r2pc_ocean:>10.4f}')
print(f'{"Delta":<14} {r2pc_all-r2all:>+8.4f} {r2pc_land-r2land:>+9.4f} {r2pc_ocean-r2ocean:>+10.4f}')

mse_all   = np.mean((pred - obs)**2)
mse_land  = np.mean((pred[landmask] - obs[landmask])**2)
mse_ocean = np.mean((pred[oceanmask] - obs[oceanmask])**2)
msepc_all   = np.mean((predpc - obs)**2)
msepc_land  = np.mean((predpc[landmask] - obs[landmask])**2)
msepc_ocean = np.mean((predpc[oceanmask] - obs[oceanmask])**2)

print()
print(f'{"Model":<14} {"MSE all":>9} {"MSE land":>10} {"MSE ocean":>11}')
print(f'{"SR-ALL":<14} {mse_all:>9.4f} {mse_land:>10.4f} {mse_ocean:>11.4f}')
print(f'{"SR-ALL-PC":<14} {msepc_all:>9.4f} {msepc_land:>10.4f} {msepc_ocean:>11.4f}')

Accuracy comparison (test split):
Model            R² all   R² land   R² ocean
SR-ALL           0.4149    0.2926     0.5089
SR-ALL-PC        0.4160    0.2925     0.5112
Delta           +0.0012   -0.0001    +0.0022

Model            MSE all   MSE land   MSE ocean
SR-ALL            2.5093     4.6023      1.6208
SR-ALL-PC         2.5043     4.6029      1.6134


In [ ]:
PCLABELS = {k:v['label'] for k,v in CONSTRAINTS.items()}
rows = []
for label,r2,constresults in [
    ('SR-ALL',[r2all,r2land,r2ocean],results),
    ('SR-ALL-PC',[r2pc_all,r2pc_land,r2pc_ocean],pcresults)]:
    row = {'Model':label,'R² all':r2[0],'R² land':r2[1],'R² ocean':r2[2]}
    for pcname in CONSTRAINTS:
        row[PCLABELS[pcname]] = constresults[pcname]['All']
    rows.append(row)

summdf = pd.DataFrame(rows).set_index('Model')
summdf.style.format({
    'R² all':'{:.4f}','R² land':'{:.4f}','R² ocean':'{:.4f}',
    **{PCLABELS[k]:'{:.1f}%' for k in CONSTRAINTS}
}).set_caption(f'Summary: accuracy and physical constraint satisfaction ({SPLIT} split, analytical derivatives)')

In [ ]:
print('Updated entry for configs.json (experiments.sr.optimizedeqs):')
print()
print(json.dumps({PCNAME:{
    'runfrom':'sr_all',
    'refcomplexity':None,
    'form':PCFORM,
    'init':{k:round(float(v),2) for k,v in pcconstants.items()},
    'color':'#8B0000',
    'description':PCLABEL}},indent=4))
print()
print('To optimize final constants on train+valid:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq')
print()
print('To generate predictions:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq --predict-only --splits test')